In [1]:
using COBREXA, GLPK, CPLEX, Tulip, DataFrames, Distributed, CSV
include("map_gene_reaction.jl")

addprocs(12)    
@everywhere using COBREXA, GLPK 
 
# MODEL_PATH = "output_model/Mouse-GEM.mat" 
model_path = "Mouse-GEM.xml"
model = load_model(model_path);
model = convert(MATModel, model)

Metabolic model of type MATModel
sparse([3010, 3012, 6028, 475, 3625, 4133, 6028, 475, 949, 3625  …  3085, 3216, 3220, 4123, 4132, 3085, 4123, 4132, 6369, 6371], [1, 1, 1, 2, 2, 2, 2, 3, 3, 3  …  12986, 12986, 12986, 12986, 12986, 12987, 12987, 12987, 12987, 12987], [1.0, -1.0, 77243.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0  …  1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0], 8454, 12987)
Number of reactions: 12987
Number of metabolites: 8454


In [2]:
# RNA_df = CSV.read("data\\GSE164051_Proj_10547_B_htseq_all_samples.txt", DataFrame)
# gene_cpm_norm.csv
RNA_df = CSV.read("data\\gene_cpm.csv", DataFrame)
RNA_df = dropmissing(RNA_df)
genes = RNA_df[!,"GeneSymbol"]
gene_exp = select(RNA_df, Not(:"GeneSymbol"))
# gene_exp = select(gene_exp, Not(:"GeneID"))
#print(size(gene_exp))
GAMMA = 2  #2 for normalised and 1 for winsorzation
gene_exp

Row,s_NC_aC_4,s_NC_aC_5,s_KD_aC_3,s_NC_IgG_1,s_KD_aC_1,s_NC_IgG_3,s_KD_aC_5,s_KD_IgG_1,s_NC_aC_1,s_KD_IgG_4,s_NC_aC_3,s_KD_IgG_5,s_NC_aC_2,s_NC_IgG_4,s_KD_aC_2,s_KD_IgG_2,s_NC_IgG_5,s_KD_aC_4,s_KD_IgG_3
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,334.494,294.911,275.837,329.787,324.042,314.748,276.18,316.371,281.069,338.688,277.568,332.074,303.951,300.178,344.4,317.927,351.853,302.347,330.555
2,41.3084,36.2298,27.2985,45.9079,36.4258,45.7449,31.8434,37.7977,36.6469,39.235,41.7003,40.9317,44.4881,46.8953,39.998,37.1991,50.6477,35.5495,42.1296
3,211.854,96.3245,57.2899,236.917,222.992,199.35,42.5319,191.649,114.542,284.399,44.8432,165.314,211.782,190.299,151.076,298.276,368.255,103.577,68.3386
4,0.271766,0.259402,0.396013,0.171572,0.277354,0.230163,0.444197,0.219435,0.109557,0.376355,0.127758,0.226768,0.352149,0.134525,0.164601,0.288976,0.172957,0.276944,0.220244
5,0.345884,0.230579,0.0528018,0.171572,0.554707,0.287704,0.277623,0.329153,0.38345,0.125452,0.15331,0.28346,0.258242,0.322859,0.329202,0.157623,0.23061,0.428004,0.157317
6,68.6826,52.8315,62.0421,57.1336,66.9578,68.79,53.2204,63.3345,56.9697,72.7619,46.2485,71.8289,67.7299,63.3611,78.3775,65.9129,77.0813,62.1361,61.7628
7,152.757,112.667,83.1628,118.949,102.089,108.09,78.956,85.6621,127.086,95.3432,105.988,84.046,106.584,121.53,91.5181,73.7678,120.667,102.771,95.0196
8,208.568,211.153,208.593,219.711,152.129,237.845,206.246,170.062,215.746,175.036,217.547,163.953,205.631,229.768,183.667,180.873,230.293,168.382,177.737
9,33.3037,33.4628,38.5717,33.9223,36.1947,38.725,38.2287,39.0046,39.1667,39.3604,36.9733,39.9679,40.1449,41.8641,39.3396,40.5618,35.2545,35.474,38.6371


In [3]:
print("Generating fluxes...")

flux_df_max = Dict{String,Any}()
flux_df_min = Dict{String,Any}()
count = 1;
for i in 1: size(gene_exp)[2]
    print('\n', "sample: ", i, '\n')
    colname = string(names(gene_exp)[i]) 
    values = gene_exp[!,i]/maximum( gene_exp[!,i])
    # values = gene_exp[!,i]

    rxnExpression = mapExpressionsToReactions(model, genes, values)
    
    const_model = constrainReactions(model, rxnExpression, GAMMA)

    try
        minFlux, maxFlux = flux_variability_analysis_dict(
            const_model,
            GLPK.Optimizer;
            workers=workers(),
            bounds = objective_bounds(0.9),
            #modifications = constrains,
            # modifications = [ # modifications are applied in order
            #     #change_objective("MAR13082"), # maximize production
            #     change_optimizer(Tulip.Optimizer), 
            #     change_sense(JuMP.MAX_SENSE),
            # ],

        )
        rxn_ids = keys(maxFlux)
        maxFluxRxn = Dict{String,Vector{Maybe{Float64}}}()
        for rxn_id in rxn_ids
            #print(rxn_id)
            ub = maxFlux[rxn_id][rxn_id]
            maxFluxRxn[rxn_id] = [ub]
        end
        flux_df_max[colname] = maxFluxRxn

        minFluxRxn = Dict{String,Vector{Maybe{Float64}}}()
        for rxn_id in rxn_ids
            #print(rxn_id)
            lb = minFlux[rxn_id][rxn_id]
            minFluxRxn[rxn_id] = [lb]
        end
        flux_df_min[colname] = minFluxRxn

        #print(maxFluxRxn)
    catch
        println("error occured")
        println(colname)
        flux_list = []
        for (key, value) in flux_df_max
            df = DataFrame(value)
            df[!, "index"] .= key
            push!(flux_list,df)
        end
        df = vcat(flux_list...)
        print("printing file ...")
        CSV.write("Fluxes\\flux_df_max_cpm.csv", df)

        flux_list = []
        for (key, value) in flux_df_min
            df = DataFrame(value)
            df[!, "index"] .= key
            push!(flux_list,df)
        end
        df = vcat(flux_list...)
        print("printing file ...")
        CSV.write("Fluxes\\flux_df_min_cpm.csv", df)

    end
    count = count + 1
end
flux_list = []
for (key, value) in flux_df_max
    df = DataFrame(value)
    df[!, "index"] .= key
    push!(flux_list,df)
end
df = vcat(flux_list...)
print("printing file ...")
CSV.write("Fluxes\\flux_df_max_cpm.csv", df)

flux_list = []
for (key, value) in flux_df_min
    df = DataFrame(value)
    df[!, "index"] .= key
    push!(flux_list,df)
end
df = vcat(flux_list...)
print("printing file ...")
CSV.write("Fluxes\\flux_df_min_cpm.csv", df)